In [2]:
import pandas as pd
import json

In [3]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
# Load the Excel file
beverage_list_path = './TCF-Coffe_App-Export_Beverage-List.csv'
impact_data_path = './TCF-Coffe_App-Export_Impact-Data.csv'
impact_description_path = './TCF-Coffe_App-Export_Impact-Description.csv'
coffee_description_path = './TCF-Coffe_App-Export_Coffee-Description.csv'

df_beverage_list = pd.read_csv(beverage_list_path,delimiter=';',encoding='windows-1252')
df_impact_data = pd.read_csv(impact_data_path,delimiter=';',encoding='windows-1252')
df_impact_description = pd.read_csv(impact_description_path,delimiter=';',encoding='utf-8')
df_coffee_description = pd.read_csv(coffee_description_path,delimiter=';',encoding='windows-1252')



df_beverage_list.head()

,Beverage ID,Retail name,Retail price,Hidden Costs,True Price,Salepoint
0,"Café, Klee",Café,1.4,0.3,1.8,"EPFL, Rolex centre, Le Klee"
1,"Cappuccino, Dallmayr",Cappuccino,1.5,2.6,4.1,"EPFL, Rolex centre, Dallmayr"
2,"Espresso, Klee",Espresso,1.4,0.3,1.8,"EPFL, Rolex centre, Le Klee"


In [5]:
def get_indicator_values(row):

    # # Find matches where df_impact_description['Indicator'] is contained in row['Indicator']
    # matched_rows = df_impact_description[
    #     df_impact_description['Indicator'].apply(lambda x: x in row['Indicator'])
    # ]
    
    # if matched_rows.empty:
    #     impact_definition = None
    #     monetisation_method = None
    # else:
    #     # Select the most specific match (longest string in df_impact_description['Indicator'])
    #     impact_definition = matched_rows.loc[matched_rows['Indicator'].str.len().idxmax(), 'Indicator definition']
    #     monetisation_method = matched_rows.loc[matched_rows['Indicator'].str.len().idxmax(), 'Monetisation method']
    
    return {
        'indicators': row['Indicator'],
        'unit': row['Unit'],
        'impactValue': row['Value'], 
        'costValue': row['Monetary Value'], 
        # 'impactDefinition': impact_definition,
        # 'monetisationMethod': monetisation_method,
        'reference': None if pd.isna(row['Reference']) else row['Reference']
    }


def calculate_impacts(group):
    # Group by stage and impact-category
    grouped_impacts = group.groupby(['Ingredient','Stage', 'Impact Category']).apply(lambda x: {
        'stage': x.iloc[0]['Stage'],
        'ingredient': x.iloc[0]['Ingredient'],
        'ingredientID': x.iloc[0]['Ingredient ID'],
        'impactCategory': x.iloc[0]['Impact Category'],
        'impactValue': x['Value'].sum(),   # Sum impact values
        'costValue': x['Monetary Value'].sum(),   # Sum cost values
        'details': x.apply(get_indicator_values, axis=1).tolist()
    }).reset_index(drop=True).tolist()

    return grouped_impacts

In [10]:
# Map Sale Point IDs
# sale_point_mapping = {name: idx + 1 for idx, name in enumerate(df_beverage_list['Salepoint'].unique())}

# Transform the data
import os


formatted_data = []

i = 0;

for _, beverage in df_beverage_list.iterrows():
    beverage_id = beverage['Beverage ID']
    label = beverage['Retail name']
    # sale_point_id = sale_point_mapping[beverage['Salepoint']]
    is_decaf = 'decaf' in beverage['Retail name'].lower()
    has_milk = any(x in beverage['Retail name'].lower() for x in ['milk', 'latte', 'Cappuccino', 'renversé'])
    milk_type = 'Dairy' if has_milk else None
    
    definition_row = df_coffee_description[df_coffee_description['Recipe'] == label]
    definition = definition_row['Definition'].iloc[0] if not definition_row.empty else "Unknown"


    impact_rows = df_impact_data[df_impact_data['Beverage ID'] == beverage_id]
    ingredient_list = impact_rows['Ingredient'].unique().tolist()
    # labels_list = [label for label in impact_rows['Labels'].unique().tolist() if label != 'none']
    # This was the real but for testing purpose here's a fake one :
    labels_list = ['Fair Trade', 'Organic', 'Rainforest Alliance'][:i]
    i += 1
    ingredient_list = impact_rows['Ingredient'].unique().tolist()

    grouped_impacts = calculate_impacts(impact_rows)
    print(grouped_impacts)

    path_impacts = './results/impacts/'+beverage_id.lower().replace(' ','_').replace(',','')+'.json'
    os.makedirs(os.path.dirname(path_impacts), exist_ok=True)
    with open(path_impacts, 'w', encoding='utf-8') as f:
        json.dump(grouped_impacts, f, indent=4, ensure_ascii=False)



    formatted_data.append({
        'serveId': beverage_id,
        'recipeId': label,
        'retailName': label,
        'retailPrice': beverage['Retail price'],
        'hiddenCost': beverage['Hidden Costs'],
        'truePrice': beverage['True Price'],
        'labels': ('#').join(labels_list),
        'isDecaf': is_decaf,
        'hasMilk': has_milk,
        'milkType': milk_type,
        'coffeeDetails': definition, 

    })

# Create the formatted DataFrame
formatted_df = pd.DataFrame(formatted_data)

# Save to CSV
output_path = './results/coffee_data.csv'
formatted_df.to_csv(output_path, index=False)



print(f"Formatted data saved to {output_path}")

[{'stage': 'Coffee Cultivation', 'ingredient': 'Coffee', 'ingredientID': 'Conventional Brazilian Coffee', 'impactCategory': 'Biodiversity', 'impactValue': np.float64(4.18979193292e-05), 'costValue': np.float64(0.00019167500129609998), 'details': [{'indicators': 'Eco-costs of land-use (Brazil), deforestation-related', 'unit': 'm²/kg', 'impactValue': 2.44534414661e-05, 'costValue': 0.0001202290663021, 'reference': 'FAOSTAT, Crops and livestock products, Yield, Coffee beans, 2022, none: Conservative, conventional, Sustainability Impact Metrics (a spin-off of Delft University of Technology), The eco-costs of land-use, 2024; World Bank, Inflation (annual %), 2024; Sustainability Impact Metrics (a spin-off of Delft University of Technology), The eco-costs of land-use, 2024; European Central Bank, Eurosystem, US dollar (USD), 2024'}, {'indicators': 'Eco-costs of land-use (Brazil), practice-related', 'unit': 'm²/kg', 'impactValue': -0.0, 'costValue': -0.0, 'reference': 'FAOSTAT, Crops and live

In [7]:
output_path= "./results/impacts_definitions.csv"
df_impact_description.to_csv(output_path, index=False)